# 06 · Portfolio Factor Model

**Purpose:** Build a multi-factor model that decomposes mining stock, ETF, and BTC derivative returns into:

1. **BTC Price Factor** — systematic BTC price exposure (beta)
2. **Hashrate Factor** — exposure to changes in network hashrate (mining difficulty)
3. **Implied Vol Factor** — exposure to BTC volatility regime changes
4. **Idiosyncratic** — company-specific alpha

Model: `R_i = α + β_BTC·R_BTC + β_HR·R_HR + β_IV·ΔIV + ε`

**Outputs:**
- Factor loading heatmap
- Efficient frontier
- Strategy comparison: mining equity vs direct BTC vs delta-neutral carry

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm

from src.data.yfinance_fetcher import fetch_prices, compute_returns
from src.data.coinmetrics import fetch_hashrate_difficulty
from src.data.binance import get_funding_rate_daily
from src.models.mining_metrics import compute_all_betas, realized_vol, sharpe_ratio
from src.utils.plotting import PLOTLY_TEMPLATE, BTC_ORANGE, MINER_COLORS
from config import MINERS, BTC_ETFS, DEFAULT_LOOKBACK_DAYS

pd.options.display.float_format = '{:.4f}'.format
print('Setup complete.')

## 1. Load and Align All Data

In [ ]:
# --- Equity prices ---
all_equity = list(MINERS.keys()) + list(BTC_ETFS.keys()) + ['BTC-USD']
prices = fetch_prices(all_equity, period='2y').dropna(how='all')
equity_returns = compute_returns(prices)

# --- Hashrate (daily % change) ---
onchain = fetch_hashrate_difficulty(days=DEFAULT_LOOKBACK_DAYS)
hashrate = onchain['hashrate']
hashrate_returns = hashrate.pct_change().rename('hashrate_return')

# --- Funding rate as vol/sentiment proxy ---
try:
    funding = get_funding_rate_daily(days=DEFAULT_LOOKBACK_DAYS)
    funding_daily = funding['daily_funding_rate'].rename('funding_rate')
except Exception:
    funding_daily = pd.Series(dtype=float, name='funding_rate')
    print('Funding rate not available — will use 2-factor model.')

# --- Align all to common dates ---
factor_df = pd.DataFrame({
    'btc_return':      equity_returns.get('BTC-USD', pd.Series()),
    'hashrate_return': hashrate_returns,
    'funding_rate':    funding_daily,
}).dropna(subset=['btc_return', 'hashrate_return'])

# Align equity returns to factor dates
equity_aligned = equity_returns.reindex(factor_df.index).dropna(how='all', axis=1)
factor_df = factor_df.reindex(equity_aligned.index)

print(f'Factor data: {len(factor_df)} trading days')
print(f'Equity tickers: {equity_aligned.columns.tolist()}')
factor_df.describe().round(4)

## 2. Multi-Factor Regression

For each asset: `R_i = α + β_BTC·R_BTC + β_HR·R_hashrate + β_fund·funding_rate + ε`

In [ ]:
def run_factor_regression(asset_returns: pd.Series, factor_df: pd.DataFrame) -> dict:
    """Run OLS multi-factor regression for a single asset."""
    factors = ['btc_return', 'hashrate_return']
    if 'funding_rate' in factor_df.columns and factor_df['funding_rate'].notna().sum() > 60:
        factors.append('funding_rate')

    combined = pd.concat([asset_returns, factor_df[factors]], axis=1).dropna()
    if len(combined) < 60:
        return {}

    y = combined.iloc[:, 0]
    X = sm.add_constant(combined[factors])
    result = sm.OLS(y, X).fit()

    out = {
        'alpha_daily':        result.params.get('const', np.nan),
        'alpha_annualized':   result.params.get('const', np.nan) * 252,
        'beta_btc':           result.params.get('btc_return', np.nan),
        'beta_hashrate':      result.params.get('hashrate_return', np.nan),
        'beta_funding':       result.params.get('funding_rate', np.nan),
        'r_squared':          result.rsquared,
        'adj_r_squared':      result.rsquared_adj,
        'n_obs':              result.nobs,
        'resid_vol_ann':      result.resid.std() * np.sqrt(252),
    }
    return out


# Run regression for each asset
factor_results = {}
for ticker in equity_aligned.columns:
    if ticker == 'BTC-USD':
        continue
    res = run_factor_regression(equity_aligned[ticker], factor_df)
    if res:
        factor_results[ticker] = res

factor_loadings = pd.DataFrame(factor_results).T
print('Multi-Factor Loadings:')
print(factor_loadings[['alpha_annualized','beta_btc','beta_hashrate','r_squared','resid_vol_ann']].round(3).to_string())

## 3. Factor Loading Heatmap

In [ ]:
# Focus on miners only
miner_loadings = factor_loadings.loc[
    factor_loadings.index.isin(list(MINERS.keys()))
].sort_values('beta_btc', ascending=False)

display_factors = ['beta_btc', 'beta_hashrate', 'alpha_annualized', 'r_squared']
z_data = miner_loadings[display_factors].values
col_labels = ['β_BTC', 'β_Hashrate', 'α (ann.)', 'R²']

fig = go.Figure(go.Heatmap(
    z=z_data,
    x=col_labels,
    y=miner_loadings.index.tolist(),
    colorscale='RdYlGn',
    text=[[f'{v:.2f}' for v in row] for row in z_data],
    texttemplate='%{text}',
    colorbar=dict(title='Loading'),
))
fig.update_layout(
    title='Mining Company Factor Loadings',
    template=PLOTLY_TEMPLATE, height=400,
)
fig.show()

## 4. BTC Price Beta vs Hashrate Beta

In [ ]:
fig = go.Figure()

for ticker, row in miner_loadings.iterrows():
    color = MINER_COLORS.get(ticker, '#888')
    fig.add_trace(go.Scatter(
        x=[row['beta_btc']],
        y=[row['beta_hashrate']],
        mode='markers+text',
        marker=dict(size=14, color=color, opacity=0.85,
                    line=dict(width=1, color='white')),
        text=[ticker],
        textposition='top center',
        name=ticker,
    ))

fig.add_vline(x=1.0, line_dash='dash', line_color='gray')
fig.add_hline(y=0.0, line_dash='dash', line_color='gray')
fig.update_layout(
    title='Mining Stocks: BTC Price Beta vs Hashrate Beta',
    xaxis_title='Beta to BTC Price',
    yaxis_title='Beta to Hashrate',
    template=PLOTLY_TEMPLATE, height=500, showlegend=False,
    annotations=[
        dict(x=2.5, y=0.8, text='High BTC & HR beta<br>(leveraged miner)', showarrow=False, font=dict(color='gray')),
        dict(x=0.5, y=-0.3, text='Low beta<br>(defensive)', showarrow=False, font=dict(color='gray')),
    ]
)
fig.show()

## 5. Efficient Frontier

In [ ]:
# Build efficient frontier for a portfolio of BTC + top 5 miners
top_miners = list(MINERS.keys())[:5]  # MARA, RIOT, CLSK, IREN, HUT
ef_tickers = top_miners + ['BTC-USD']
ef_tickers = [t for t in ef_tickers if t in equity_aligned.columns]

ef_returns = equity_aligned[ef_tickers].dropna()
mu = ef_returns.mean() * 252          # annualized expected return
cov = ef_returns.cov() * 252          # annualized covariance

n_assets = len(ef_tickers)
n_simulations = 5000
rng = np.random.default_rng(42)

sim_returns, sim_vols, sim_sharpes, sim_weights = [], [], [], []

for _ in range(n_simulations):
    w = rng.dirichlet(np.ones(n_assets))
    port_ret = float(w @ mu)
    port_vol = float(np.sqrt(w @ cov @ w))
    port_sr  = (port_ret - 0.05) / port_vol
    sim_returns.append(port_ret)
    sim_vols.append(port_vol)
    sim_sharpes.append(port_sr)
    sim_weights.append(w)

sim_returns = np.array(sim_returns)
sim_vols    = np.array(sim_vols)
sim_sharpes = np.array(sim_sharpes)

max_sr_idx = np.argmax(sim_sharpes)
min_vol_idx = np.argmin(sim_vols)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sim_vols * 100, y=sim_returns * 100,
    mode='markers',
    marker=dict(color=sim_sharpes, colorscale='Viridis', size=3, opacity=0.5,
                colorbar=dict(title='Sharpe Ratio')),
    name='Simulated Portfolios',
))

# Highlight key portfolios
for idx, label, color in [
    (max_sr_idx, 'Max Sharpe', '#F7931A'),
    (min_vol_idx, 'Min Vol', '#27AE60'),
]:
    fig.add_trace(go.Scatter(
        x=[sim_vols[idx] * 100], y=[sim_returns[idx] * 100],
        mode='markers+text', text=[label], textposition='top right',
        marker=dict(size=16, color=color, symbol='star'),
        name=label,
    ))

# Individual assets
for i, t in enumerate(ef_tickers):
    fig.add_trace(go.Scatter(
        x=[float(np.sqrt(cov.iloc[i, i])) * 100],
        y=[float(mu.iloc[i]) * 100],
        mode='markers+text', text=[t], textposition='top center',
        marker=dict(size=12, color=MINER_COLORS.get(t, '#888'),
                    symbol='diamond', line=dict(width=1, color='white')),
        name=t, showlegend=False,
    ))

fig.update_layout(
    title='Efficient Frontier: Top Miners + BTC (Monte Carlo)',
    xaxis_title='Annualized Volatility (%)',
    yaxis_title='Annualized Return (%)',
    template=PLOTLY_TEMPLATE, height=550,
)
fig.show()

print('\nMax Sharpe Portfolio weights:')
max_sr_w = sim_weights[max_sr_idx]
for t, w in zip(ef_tickers, max_sr_w):
    print(f'  {t:<8}: {w*100:.1f}%')
print(f'  Return: {sim_returns[max_sr_idx]*100:.1f}%  Vol: {sim_vols[max_sr_idx]*100:.1f}%  Sharpe: {sim_sharpes[max_sr_idx]:.2f}')

## 6. Strategy Comparison

Compare three approaches to BTC exposure:
1. **Direct BTC** — hold spot BTC
2. **Equal-weight miners** — equal allocation to top 5 miners
3. **BTC + perp carry** — hold BTC spot, short perp (collect funding rate, delta-neutral in futures)

In [ ]:
# Strategy 1: Direct BTC
strat_btc = equity_aligned['BTC-USD'].rename('Direct BTC')

# Strategy 2: Equal-weight top 5 miners
miner_cols = [t for t in top_miners if t in equity_aligned.columns]
strat_miners = equity_aligned[miner_cols].mean(axis=1).rename('EW Top-5 Miners')

# Strategy 3: BTC + funding carry (long spot BTC, collect funding on short perp)
# Simplified: BTC return + daily funding rate (collected as short perp)
if 'funding_rate' in factor_df.columns and factor_df['funding_rate'].notna().sum() > 60:
    funding_aligned = factor_df['funding_rate'].reindex(equity_aligned.index)
    strat_carry = (equity_aligned['BTC-USD'] + funding_aligned).rename('BTC + Carry')
else:
    strat_carry = equity_aligned['BTC-USD'].rename('BTC + Carry (no funding data)')  

# Combine and compute cumulative returns
strategies = pd.concat([strat_btc, strat_miners, strat_carry], axis=1).dropna()
cumulative = (1 + strategies).cumprod() * 100

fig = go.Figure()
strat_colors = [BTC_ORANGE, '#E74C3C', '#27AE60']
for col, color in zip(cumulative.columns, strat_colors):
    fig.add_trace(go.Scatter(
        x=cumulative.index, y=cumulative[col],
        name=col, mode='lines', line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Strategy Comparison: Direct BTC vs Miners vs BTC Carry',
    yaxis_title='Growth of $100',
    template=PLOTLY_TEMPLATE, height=450, hovermode='x unified',
)
fig.show()

# Performance stats
print(f'\n{'Strategy':<30} {'Total Return':>14} {'Ann. Vol':>10} {'Sharpe':>8}')
print('-' * 66)
for col in strategies.columns:
    total_ret = (1 + strategies[col]).prod() - 1
    ann_vol   = strategies[col].std() * np.sqrt(252)
    sr        = sharpe_ratio(strategies[col])
    print(f'{col:<30} {total_ret*100:>13.1f}% {ann_vol*100:>9.1f}% {sr:>8.2f}')

## 7. Rolling Factor Exposures

In [ ]:
# Rolling 90-day beta to BTC for each miner
window = 90
rolling_betas = {}

for ticker in miner_cols:
    betas = []
    dates = []
    combined = pd.concat([equity_aligned[ticker], factor_df['btc_return']], axis=1).dropna()

    for i in range(window, len(combined)):
        chunk = combined.iloc[i-window:i]
        X = sm.add_constant(chunk['btc_return'])
        try:
            res = sm.OLS(chunk.iloc[:, 0], X).fit()
            betas.append(res.params['btc_return'])
        except Exception:
            betas.append(np.nan)
        dates.append(combined.index[i])

    rolling_betas[ticker] = pd.Series(betas, index=dates)

fig = go.Figure()
for ticker, series in rolling_betas.items():
    color = MINER_COLORS.get(ticker, '#888')
    fig.add_trace(go.Scatter(
        x=series.index, y=series.values,
        name=ticker, mode='lines',
        line=dict(color=color, width=1.5),
    ))

fig.add_hline(y=1.0, line_dash='dash', line_color='white', line_width=0.5)
fig.update_layout(
    title=f'Rolling {window}-Day Beta to BTC by Miner',
    yaxis_title='Beta to BTC', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=450, hovermode='x unified',
)
fig.show()

## Summary

### Key Findings

1. **BTC beta dominates**: Mining stocks have BTC price beta of 1.5–3.0x, making them highly leveraged BTC plays

2. **Hashrate beta is positive but smaller**: Miners benefit from rising hashrate (indicates industry health) but rising hashrate also increases competition, compressing hashprice

3. **High idiosyncratic volatility**: Despite high R², significant company-specific risk remains (operational, leverage, treasury strategy)

4. **Equal-weight miner portfolio** can outperform direct BTC in bull markets (higher beta) but suffers more in bear markets

5. **BTC + carry strategy** delivers BTC returns plus the funding rate premium when perpetual longs are dominant

### Factor Summary

| Asset Class | BTC Beta | Hashrate Beta | Notes |
|-------------|----------|---------------|-------|
| Direct BTC | 1.0 | ~0 | Benchmark |
| Large miners (MARA, RIOT) | 2.0–3.0 | 0.3–0.7 | Leveraged BTC + hashrate |
| MSTR | 3.0+ | ~0 | Pure BTC treasury leverage |
| WGMI ETF | 1.5–2.0 | 0.2–0.5 | Diversified mining |
| BTC ETF (IBIT) | ~1.0 | ~0 | Spot BTC proxy |
| BTC perp (funding carry) | 1.0 | ~0 | BTC + yield premium |